In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
# Tratamiento de Datos
import pandas as pd
import numpy as np
from IPython.display import display

# Visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns

# Para que se muestren todas las columnas al inspeccionar los DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)

In [3]:
from framework import sp_carga_exploracion as sc
from framework import sp_limpieza_transformacion as sl

In [4]:
ruta = (r"../data/raw/dataset_estudiantes.csv")
df = sc.leer_csv(ruta)

CSV CARGADO
Archivo: ../data/raw/dataset_estudiantes.csv
Filas:   1000
Columnas: 11


In [5]:
df_l = df.copy()

#### 0. CALIDAD DEL DATASET

In [6]:
sl.reporte_calidad(df_l)

,tipo,ejemplos
horas_estudio_semanal,float64,"[8.957475998038344, 11.042524001961656, 4.510776081457513]"
nota_anterior,float64,"[48.83060095398218, 80.82570664326124, 90.38369427718838]"
tasa_asistencia,float64,"[86.64018172631499, 83.44965544904211, 74.62360717097394]"
horas_sueno,float64,"[6.675694179152657, 4.616843562140823, 7.755246085587813]"
edad,int64,"[25, 18, 23]"
nivel_dificultad,str,"[Fácil, Difícil, Medio]"
tiene_tutor,str,"[Sí, No]"
horario_estudio_preferido,str,"[Tarde, Mañana, Noche]"
estilo_aprendizaje,str,"[Lectura/Escritura, Visual, Auditivo]"
nota_final,float64,"[84.4, 72.0, 80.0]"


#### 1. NORMALIZAR COLUMNAS

Aunque los nombres de las columnas vienen correctamente segun forma `snake_case` vamos a usar la funcion `limpiar_columnas` para estar seguros que estan correctamente.

In [7]:
df_l = sl.limpiar_columnas(df_l)

In [8]:
df_l.columns.tolist()

['horas_estudio_semanal',
 'nota_anterior',
 'tasa_asistencia',
 'horas_sueno',
 'edad',
 'nivel_dificultad',
 'tiene_tutor',
 'horario_estudio_preferido',
 'estilo_aprendizaje',
 'nota_final',
 'aprobado']

#### 2. CORREGIR TIPOS DE DATOS

Observación a valorar en preprocesamientoe de Fase 4: nivel_dificultad → category

`nivel_dificultad` tiene un orden natural (Fácil < Medio < Difícil), a diferencia de `tiene_tutor` (binaria, sin orden) o `estilo_aprendizaje/horario_estudio_preferido` (categorías sin jerarquía entre sí, Visual no es "más" ni "menos" que Auditivo).

##### REEMPLAZAR CARACTERES / POR _

In [9]:
df_l = sl.reemplazar_caracter(df_l, ['estilo_aprendizaje'], '/', '_') 

#### 3. TRATAR VALORES NULOS

In [10]:
sl.buscar_nulos(df_l)

,nulos,%_nulos
horas_sueno,150,15.0
horario_estudio_preferido,100,10.0
estilo_aprendizaje,50,5.0


In [11]:
sl.columnas_con_nulos(df_l)

['horas_sueno', 'horario_estudio_preferido', 'estilo_aprendizaje']

In [12]:
columnas = ['horas_sueno', 'horario_estudio_preferido', 'estilo_aprendizaje']

In [13]:
sl.estadisticas_nulos(df_l,columnas)

RECORDATORIO PARA VALORAR LA IMPUTACIÓN
• Media ≈ mediana: la media podría ser razonable.
• Media muy distinta de mediana: valorar la mediana.
• Std muy alta: posible presencia de outliers.
• Variables categóricas: usar la moda.


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
horas_sueno,850.0,NaN,NaN,NaN,7.00767,1.44479,4.0,5.995341,7.020701,8.018834,10.0
horario_estudio_preferido,900,3,Noche,344,NaN,NaN,NaN,NaN,NaN,NaN,NaN
estilo_aprendizaje,950,4,Visual,363,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df_l[['horario_estudio_preferido','estilo_aprendizaje']].mode()

,horario_estudio_preferido,estilo_aprendizaje
0,Noche,Visual


In [15]:
df_l = sl.imputar_media(df_l,'horas_sueno')

In [16]:
col_moda = ['horario_estudio_preferido','estilo_aprendizaje']

In [17]:
df_l = sl.imputar_moda(df_l,col_moda)

In [18]:
df_l.isna().sum().sum()

np.int64(0)

#### 3.5. TRATAR COLUMNAS BINARIAS 

In [19]:
df_l = sl.crear_binaria_ml(df_l, 'tiene_tutor', {'No': 0, 'Sí': 1})

✓ Columna creada: "tiene_tutor_ml" — todos los valores mapeados correctamente.


Observaciones: <br>
`tiene_tutor` es una columna categorica sin nulos, que convertimos a binaria para utilizarla en posteriores consultas, se crea una nueva columnas `tiene_tutor_ml`


In [20]:
df_l.sample(2).T

,496,43
horas_estudio_semanal,7.777331,6.193237
nota_anterior,86.330838,65.15602
tasa_asistencia,100.0,62.578021
horas_sueno,7.00767,7.828805
edad,18,29
nivel_dificultad,Medio,Medio
tiene_tutor,No,No
horario_estudio_preferido,Mañana,Tarde
estilo_aprendizaje,Auditivo,Visual
nota_final,74.7,64.9


#### 4. ELIMINAR DUPLICADOS

In [21]:
sl.buscar_duplicados(df_l)

0

Observaciones:  

Confirmamos que no existen duplicados.

#### 5. LIMPIAR TEXTO INCONSISTENTE

##### *TRATAMIENTO COLUMNAS CATEGORICAS*

In [22]:
col_cat = df_l.select_dtypes(include=['string','category'])

for cols in col_cat:
    print("-"*50)
    print(cols.upper())
    print(df_l[cols].unique())
    print("-"*50)      




--------------------------------------------------
NIVEL_DIFICULTAD
<ArrowStringArray>
['Fácil', 'Difícil', 'Medio']
Length: 3, dtype: str
--------------------------------------------------
--------------------------------------------------
TIENE_TUTOR
<ArrowStringArray>
['Sí', 'No']
Length: 2, dtype: str
--------------------------------------------------
--------------------------------------------------
HORARIO_ESTUDIO_PREFERIDO
<ArrowStringArray>
['Tarde', 'Mañana', 'Noche']
Length: 3, dtype: str
--------------------------------------------------
--------------------------------------------------
ESTILO_APRENDIZAJE
<ArrowStringArray>
['Lectura_Escritura', 'Visual', 'Auditivo', 'Kinestésico']
Length: 4, dtype: str
--------------------------------------------------


In [23]:
sl.ver_valores_string(df_l)


--- nivel_dificultad ---
<ArrowStringArray>
['Fácil', 'Difícil', 'Medio']
Length: 3, dtype: str

--- tiene_tutor ---
<ArrowStringArray>
['Sí', 'No']
Length: 2, dtype: str

--- horario_estudio_preferido ---
<ArrowStringArray>
['Tarde', 'Mañana', 'Noche']
Length: 3, dtype: str

--- estilo_aprendizaje ---
<ArrowStringArray>
['Lectura_Escritura', 'Visual', 'Auditivo', 'Kinestésico']
Length: 4, dtype: str


In [24]:
cols_texto = ['nivel_dificultad', 'tiene_tutor', 'horario_estudio_preferido', 'estilo_aprendizaje']

LIMPIAR TEXTO <br>
Convertir texto a misnusculas, quirar espacios y reemplazarlos por _

In [25]:
df_l = sl.limpiar_texto(df_l, cols_texto)

ELIMINAR ACENTOS

In [26]:
df_l = sl.eliminar_acentos(df_l, cols_texto)

##### *TRATAMIENTO COLUMNAS NUMERICAS*

Redondear valores de float.

In [27]:
cols_float = ['horas_estudio_semanal', 'nota_anterior', 'tasa_asistencia', 'horas_sueno', 'nota_final']

df_l[cols_float] = df_l[cols_float].round(2)

In [28]:
sl.ver_valores_string(df_l)


--- nivel_dificultad ---
<ArrowStringArray>
['facil', 'dificil', 'medio']
Length: 3, dtype: str

--- tiene_tutor ---
<ArrowStringArray>
['si', 'no']
Length: 2, dtype: str

--- horario_estudio_preferido ---
<ArrowStringArray>
['tarde', 'manana', 'noche']
Length: 3, dtype: str

--- estilo_aprendizaje ---
<ArrowStringArray>
['lectura_escritura', 'visual', 'auditivo', 'kinestesico']
Length: 4, dtype: str


#### 6. ELIMINAR VARIABLES IRRELEVANTES

In [29]:
sl.buscar_constantes(df_l)

[]

In [30]:
sl.buscar_columnas_vacias(df_l)

[]

In [31]:
sl.buscar_ids(df_l)

[]

In [32]:
sl.buscar_alta_cardinalidad(df_l)

['horas_estudio_semanal', 'nota_anterior']

`horas_estudio_semanal` y `nota_anterior` son variables numéricas continuas, es totalmente normal y esperable que casi cada estudiante tenga un valor ligeramente distinto, por lo que seria un Falso Positivo, se dejan tal cual en df_l

#### 6.5. VALIDAR COHERENCIA ENTRE COLUMNAS (REGLAS DE NEGOCIO)

In [33]:
df_l.groupby('aprobado')['nota_final'].agg(['min','max'])

,min,max
aprobado,,
0,30.0,59.9
1,60.0,100.0


In [34]:
mask = ((df_l['aprobado'] == 1) & (df_l['nota_final'] < 60)) | ((df_l['aprobado'] == 0) & (df_l['nota_final'] >= 60))
mask.sum()

np.int64(0)

Recuerda por qué esto sí es una regla de negocio válida (repasando el criterio de siempre: "¿esto no puede pasar nunca por lógica, o solo suele no pasar?"): el propio enunciado del proyecto lo dice explícitamente — "aprobado: 1 si la nota es ≥ 60, 0 en caso contrario". No es una tendencia, es una definición exacta. Por eso, cualquier fila que la incumpla sería un error real de los datos (o una señal de que la regla no es exactamente esa), no una casualidad a interpretar con cuidado como nos pasó con contactos_prev.

Ejecútalo y pásame el resultado — ya adelanté antes con el groupby('aprobado')['nota_final'].agg(['min','max']) que el corte parecía limpio en 60 (min de aprobado=1 era 60.0, max de aprobado=0 era 59.9), así que espero que salga en 0, pero confirmémoslo con la regla explícita en vez de dar por bueno lo que vimos de refilón al principio.

#### 7. DETECTAR Y TRATAR OUTLIERS

In [35]:
sl.detectar_outliers_todas(df_l)

,columna,outliers,%_outliers,limite_inferior,limite_superior
0,aprobado,102,10.2,1.000,1.000
1,nota_final,5,0.5,45.088,97.588
2,horas_estudio_semanal,4,0.4,-3.490,23.490
3,tasa_asistencia,4,0.4,21.050,128.950


🔴 aprobado — descartar, no es un outlier real

Mismo caso que impago/hipoteca en el proyecto anterior: es una variable binaria (0/1), y la mayoría son 1 (898 de 1000, un 89,8%). El IQR calcula Q1=Q3=1 (porque la mayoría de valores son 1), así que cualquier fila con 0 (el 10,2% restante) se marca como "outlier" — pero simplemente es la categoría minoritaria de una binaria, no un valor extremo. El método IQR no aplica aquí, igual que aprendimos entonces.

In [36]:
sl.detectar_outliers_iqr(df_l, 'nota_final')


Columna: nota_final
Límite inferior: 45.088  |  Límite superior: 97.588
Outliers encontrados: 5 (0.50%)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,tiene_tutor,horario_estudio_preferido,estilo_aprendizaje,nota_final,aprobado,tiene_tutor_ml
323,1.00,37.05,24.76,5.72,18,medio,no,noche,kinestesico,40.0,0,0
579,1.00,30.00,64.24,7.97,29,facil,no,noche,visual,30.0,0,0
606,16.61,100.00,100.00,8.75,27,medio,no,noche,lectura_escritura,99.2,1,0
822,2.43,75.36,84.84,7.16,27,facil,no,manana,visual,44.9,0,0
927,14.83,100.00,72.81,7.65,18,medio,no,noche,visual,100.0,1,0


In [37]:
outliers_nf = sl.detectar_outliers_iqr(df_l, 'nota_final')
print(outliers_nf[['nota_final', 'aprobado', 'horas_estudio_semanal']])

Columna: nota_final
Límite inferior: 45.088  |  Límite superior: 97.588
Outliers encontrados: 5 (0.50%)
     nota_final  aprobado  horas_estudio_semanal
323        40.0         0                   1.00
579        30.0         0                   1.00
606        99.2         1                  16.61
822        44.9         0                   2.43
927       100.0         1                  14.83


In [38]:
sl.detectar_outliers_iqr(df_l, 'horas_estudio_semanal')


Columna: horas_estudio_semanal
Límite inferior: -3.490  |  Límite superior: 23.490
Outliers encontrados: 4 (0.40%)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,tiene_tutor,horario_estudio_preferido,estilo_aprendizaje,nota_final,aprobado,tiene_tutor_ml
384,24.39,94.96,79.84,7.01,28,facil,si,noche,visual,71.5,1,1
594,25.00,73.32,82.74,8.10,22,medio,si,noche,auditivo,77.7,1,1
695,23.74,76.05,58.40,6.55,24,facil,si,manana,auditivo,72.5,1,1
945,25.00,90.87,100.00,7.90,23,medio,si,tarde,lectura_escritura,86.7,1,1


In [39]:
outliers_hes = sl.detectar_outliers_iqr(df_l, 'horas_estudio_semanal')
print(outliers_hes[['horas_estudio_semanal', 'nota_final']])

Columna: horas_estudio_semanal
Límite inferior: -3.490  |  Límite superior: 23.490
Outliers encontrados: 4 (0.40%)
     horas_estudio_semanal  nota_final
384                  24.39        71.5
594                  25.00        77.7
695                  23.74        72.5
945                  25.00        86.7


In [40]:
sl.detectar_outliers_iqr(df_l, 'tasa_asistencia')

Columna: tasa_asistencia
Límite inferior: 21.050  |  Límite superior: 128.950
Outliers encontrados: 4 (0.40%)


,horas_estudio_semanal,nota_anterior,tasa_asistencia,horas_sueno,edad,nivel_dificultad,tiene_tutor,horario_estudio_preferido,estilo_aprendizaje,nota_final,aprobado,tiene_tutor_ml
603,10.21,60.22,20.00,9.57,19,medio,no,manana,visual,65.7,1,0
630,9.07,60.13,20.00,8.06,29,medio,si,tarde,auditivo,64.5,1,1
744,6.21,76.01,20.05,4.81,23,facil,si,manana,visual,65.0,1,1
974,1.00,30.22,20.00,4.34,23,medio,no,noche,visual,50.8,0,0


In [41]:
outliers_ta = sl.detectar_outliers_iqr(df_l, 'tasa_asistencia')
print(outliers_ta[['tasa_asistencia', 'nota_final']])

Columna: tasa_asistencia
Límite inferior: 21.050  |  Límite superior: 128.950
Outliers encontrados: 4 (0.40%)
     tasa_asistencia  nota_final
603            20.00        65.7
630            20.00        64.5
744            20.05        65.0
974            20.00        50.8


### Outliers (Fase 2, paso 7) — ninguno requiere tratamiento

Se detectaron outliers IQR en nota_final (5), horas_estudio_semanal (4)
y tasa_asistencia (4). Revisados caso por caso: todos son coherentes
con el resto de variables del mismo estudiante (ej. nota_final muy
baja + nota_anterior muy baja + pocas horas de estudio). Se conservan
sin modificar — son valores extremos reales, no errores de captura.

aprobado se excluye del análisis de outliers por ser variable binaria,
donde el criterio IQR no aplica.

#### 8. VALIDACIÓN FINAL Y GUARDADO

In [42]:
sl.guardar_csv(df_l,'../data/processed/02_datos_limpios.csv')

ARCHIVO GUARDADO
Ruta:     ../data/processed/02_datos_limpios.csv
Filas:    1000
Columnas: 12
